# Lithium Supply Chain — Network Model with Flow Visualization
**Stages:** Mine → Processing | **Regions:** US · China · Australia | **Years:** 2025–2035


In [ ]:
!pip install gurobipy openpyxl networkx matplotlib -q

In [ ]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
pd.set_option('display.float_format', '{:,.1f}'.format)
print('Imports OK')

In [ ]:
R            = ['US', 'China', 'Australia']
Y            = list(range(2025, 2036))
MINE_TECHS   = ['spod', 'brine']
PROC_TECHS   = ['loh',  'lce']
MINE_TO_PROC = [('spod','loh'), ('brine','lce')]
STAGES       = ['mine', 'proc']
N = (  [('mine', t, r) for t in MINE_TECHS for r in R]
     + [('proc', t, r) for t in PROC_TECHS for r in R])
print(f'Nodes: {len(N)}')

In [ ]:
alpha  = {'US':1.3, 'China':1.0, 'Australia':1.1}
imp_cf = {'US':0.5, 'China':2.0, 'Australia':0.3}
invR   = 0.01
prod_cost = {
    ('mine','spod', 'US'):500,  ('mine','spod', 'China'):400,  ('mine','spod', 'Australia'):350,
    ('mine','brine','US'):450,  ('mine','brine','China'):380,  ('mine','brine','Australia'):320,
    ('proc','loh',  'US'):800,  ('proc','loh',  'China'):600,  ('proc','loh',  'Australia'):700,
    ('proc','lce',  'US'):750,  ('proc','lce',  'China'):550,  ('proc','lce',  'Australia'):650,
}
imp_cost = {
    ('mine','spod', 'US'):80,   ('mine','spod', 'China'):60,   ('mine','spod', 'Australia'):50,
    ('mine','brine','US'):80,   ('mine','brine','China'):60,   ('mine','brine','Australia'):50,
    ('proc','loh',  'US'):120,  ('proc','loh',  'China'):100,  ('proc','loh',  'Australia'):90,
    ('proc','lce',  'US'):120,  ('proc','lce',  'China'):100,  ('proc','lce',  'Australia'):90,
}
fixed_cost = {
    ('mine','spod'):5000, ('mine','brine'):4500,
    ('proc','loh') :8000, ('proc','lce') :7500,
}
demand = {
    ('proc','loh','US'):        {y: 10+(y-2025)*1   for y in Y},
    ('proc','loh','China'):     {y: 40+(y-2025)*4   for y in Y},
    ('proc','loh','Australia'): {y:  5+(y-2025)*0.5 for y in Y},
    ('proc','lce','US'):        {y:  8+(y-2025)*1   for y in Y},
    ('proc','lce','China'):     {y: 30+(y-2025)*3   for y in Y},
    ('proc','lce','Australia'): {y:  4+(y-2025)*0.5 for y in Y},
}
init_cap = {
    ('mine','spod', 'US'): 50,  ('mine','spod', 'China'):150,  ('mine','spod', 'Australia'):200,
    ('mine','brine','US'): 30,  ('mine','brine','China'):120,  ('mine','brine','Australia'): 80,
    ('proc','loh',  'US'): 15,  ('proc','loh',  'China'): 60,  ('proc','loh',  'Australia'): 10,
    ('proc','lce',  'US'): 12,  ('proc','lce',  'China'): 45,  ('proc','lce',  'Australia'):  8,
}
CONVERSION    = {('spod','loh'): 0.29, ('brine','lce'): 0.19}
MAX_CAP       = {'mine': 50, 'proc': 30}
MAX_NEW_UNITS = 10
print('Parameters loaded.')

In [ ]:
within_arcs = []
for (t_m, t_p) in MINE_TO_PROC:
    for r in R:
        within_arcs.append(('mine', t_m, r, 'proc', t_p, r))

trade_arcs = []
for s in STAGES:
    techs = MINE_TECHS if s=='mine' else PROC_TECHS
    for t in techs:
        for r_from in R:
            for r_to in R:
                if r_from != r_to:
                    trade_arcs.append((s, t, r_from, s, t, r_to))

WA = list(range(len(within_arcs)))
TA = list(range(len(trade_arcs)))
w_in={n:[] for n in N}; w_out={n:[] for n in N}
t_in={n:[] for n in N}; t_out={n:[] for n in N}
for idx,arc in enumerate(within_arcs):
    w_out[(arc[0],arc[1],arc[2])].append(idx)
    w_in[ (arc[3],arc[4],arc[5])].append(idx)
for idx,arc in enumerate(trade_arcs):
    t_out[(arc[0],arc[1],arc[2])].append(idx)
    t_in[ (arc[3],arc[4],arc[5])].append(idx)
print(f'Within arcs: {len(within_arcs)}  |  Trade arcs: {len(trade_arcs)}')

In [ ]:
m   = gp.Model('Lithium_2Stage_3Region')
m.Params.OutputFlag = 0   # suppress solver log for cleaner output
f_w = m.addVars(WA, Y, lb=0, name='f_w')
f_t = m.addVars(TA, Y, lb=0, name='f_t')
prod      = {(s,t,r,y): m.addVar(lb=0, name=f'prod_{s}_{t}_{r}_{y}')      for (s,t,r) in N for y in Y}
capacity  = {(s,t,r,y): m.addVar(lb=0, name=f'cap_{s}_{t}_{r}_{y}')       for (s,t,r) in N for y in Y}
new_cap   = {(s,t,r,y): m.addVar(lb=0, name=f'newcap_{s}_{t}_{r}_{y}')    for (s,t,r) in N for y in Y}
invest    = {(s,t,r,y): m.addVar(lb=0, vtype=GRB.INTEGER, name=f'inv_{s}_{t}_{r}_{y}') for (s,t,r) in N for y in Y}
stockpile = {(s,t,r,y): m.addVar(lb=0, name=f'stk_{s}_{t}_{r}_{y}')       for (s,t,r) in N for y in Y}
m.update()
print(f'Variables: {m.NumVars}')

In [ ]:
for (s,t,r) in N:
    n=(s,t,r)
    for y in Y:
        carry  = stockpile[s,t,r,y-1] if y>Y[0] else 0
        inflow = (prod[s,t,r,y]
                  + quicksum(f_w[a,y] for a in w_in[n])
                  + quicksum(f_t[a,y] for a in t_in[n]) + carry)
        outflow= (quicksum(f_w[a,y] for a in w_out[n])
                  + quicksum(f_t[a,y] for a in t_out[n])
                  + stockpile[s,t,r,y])
        m.addConstr(inflow==outflow, name=f'bal_{s}_{t}_{r}_{y}')
for (s,t,r),d_dict in demand.items():
    n=(s,t,r)
    for y in Y:
        m.addConstr(prod[s,t,r,y]+quicksum(f_t[a,y] for a in t_in[n])-quicksum(f_t[a,y] for a in t_out[n])>=d_dict[y], name=f'dem_{s}_{t}_{r}_{y}')
for wa_idx,arc in enumerate(within_arcs):
    s_f,t_f,r_f,s_t,t_t,r_t=arc
    ratio=CONVERSION[(t_f,t_t)]
    for y in Y:
        m.addConstr(f_w[wa_idx,y]<=ratio*prod[s_f,t_f,r_f,y], name=f'conv_{wa_idx}_{y}')
for ta_idx,arc in enumerate(trade_arcs):
    s_f,t_f,r_f,s_t,t_t,r_t=arc
    for y in Y:
        m.addConstr(f_t[ta_idx,y]<=prod[s_f,t_f,r_f,y], name=f'explim_{ta_idx}_{y}')
for (s,t,r) in N:
    cap_max=MAX_CAP[s]
    for y in Y:
        m.addConstr(prod[s,t,r,y]<=capacity[s,t,r,y], name=f'prodcap_{s}_{t}_{r}_{y}')
        m.addConstr(new_cap[s,t,r,y]<=cap_max*invest[s,t,r,y], name=f'invcap_{s}_{t}_{r}_{y}')
        if y==Y[0]:
            m.addConstr(capacity[s,t,r,y]==init_cap.get((s,t,r),0), name=f'initcap_{s}_{t}_{r}')
        else:
            m.addConstr(capacity[s,t,r,y]==capacity[s,t,r,y-1]+new_cap[s,t,r,y], name=f'capaccum_{s}_{t}_{r}_{y}')
for s in STAGES:
    sn=[(s2,t,r) for (s2,t,r) in N if s2==s]
    for y in Y:
        m.addConstr(quicksum(invest[s2,t,r,y] for (s2,t,r) in sn)<=MAX_NEW_UNITS, name=f'maxunits_{s}_{y}')
print(f'Constraints: {m.NumConstrs}')

In [ ]:
obj=gp.LinExpr()
for (s,t,r) in N:
    pc=prod_cost.get((s,t,r),500); fc=fixed_cost.get((s,t),5000)
    for y in Y:
        obj+=alpha[r]*pc*prod[s,t,r,y]
        obj+=fc*invest[s,t,r,y]
        obj+=invR*pc*stockpile[s,t,r,y]
for ta_idx,arc in enumerate(trade_arcs):
    s_f,t_f,r_f,s_t,t_t,r_t=arc
    ic=imp_cost.get((s_t,t_t,r_t),100)
    for y in Y:
        obj+=imp_cf[r_t]*ic*f_t[ta_idx,y]
m.setObjective(obj,GRB.MINIMIZE)
print('Objective set.')

In [ ]:
m.Params.TimeLimit=120; m.Params.MIPGap=0.01
m.optimize()
status_map={2:'OPTIMAL',3:'INFEASIBLE',9:'TIME_LIMIT'}
print(f'Status: {status_map.get(m.Status,m.Status)}')
if m.Status in(GRB.OPTIMAL,GRB.TIME_LIMIT):
    print(f'Objective: ${m.ObjVal:,.0f}')
elif m.Status==GRB.INFEASIBLE:
    m.computeIIS()
    for c in m.getConstrs():
        if c.IISConstr: print(' Infeasible:',c.ConstrName)

## Results — Build DataFrames

In [ ]:
assert m.Status in(GRB.OPTIMAL,GRB.TIME_LIMIT),'No solution found.'

# Production table
prod_rows=[]
for (s,t,r) in N:
    for y in Y:
        pv=prod[s,t,r,y].X; cv=capacity[s,t,r,y].X; iv=invest[s,t,r,y].X
        prod_rows.append({'Year':y,'Stage':s,'Tech':t,'Region':r,
                          'Production_kt':round(pv,1),'Capacity_kt':round(cv,1),'New_Units':int(round(iv))})
df_prod=pd.DataFrame(prod_rows)

# Within-region flow table
w_rows=[]
for wa_idx,arc in enumerate(within_arcs):
    s_f,t_f,r_f,s_t,t_t,r_t=arc
    for y in Y:
        v=f_w[wa_idx,y].X
        w_rows.append({'Year':y,'From_Stage':s_f,'From_Tech':t_f,'Region':r_f,
                       'To_Stage':s_t,'To_Tech':t_t,'Flow_kt':round(v,1)})
df_within=pd.DataFrame(w_rows)

# Trade flow table
t_rows=[]
for ta_idx,arc in enumerate(trade_arcs):
    s_f,t_f,r_f,s_t,t_t,r_t=arc
    for y in Y:
        v=f_t[ta_idx,y].X
        t_rows.append({'Year':y,'Stage':s_f,'Tech':t_f,
                       'Exporter':r_f,'Importer':r_t,'Flow_kt':round(v,1)})
df_trade=pd.DataFrame(t_rows)

# Cost breakdown table
cost_rows=[]
for r in R:
    rn=[(s,t,r2) for (s,t,r2) in N if r2==r]
    pc =sum(alpha[r]*prod_cost.get((s,t,r),500)*prod[s,t,r,y].X  for (s,t,r2) in rn for y in Y)
    ic =sum(fixed_cost.get((s,t),5000)*invest[s,t,r,y].X         for (s,t,r2) in rn for y in Y)
    imc=sum(imp_cf[r]*imp_cost.get((s_t,t_t,r),100)*f_t[ta_idx,y].X
            for ta_idx,arc in enumerate(trade_arcs)
            for (s_f,t_f,r_f,s_t,t_t,r_t) in [arc] if r_t==r for y in Y)
    cost_rows.append({'Region':r,'Production_$':round(pc),'Investment_$':round(ic),
                      'Import_$':round(imc),'Total_$':round(pc+ic+imc)})
df_cost=pd.DataFrame(cost_rows)
print('DataFrames ready: df_prod, df_within, df_trade, df_cost')

## Export Results to Excel

This writes all four result tables into one `.xlsx` file with separate sheets, then triggers a download in Colab.


In [ ]:
with pd.ExcelWriter('lithium_results.xlsx', engine='openpyxl') as writer:
    df_prod.to_excel(  writer, sheet_name='Production',    index=False)
    df_within.to_excel(writer, sheet_name='Within_Flows',  index=False)
    df_trade.to_excel( writer, sheet_name='Trade_Flows',   index=False)
    df_cost.to_excel(  writer, sheet_name='Cost_Breakdown',index=False)

print('lithium_results.xlsx saved.')

# Download to your computer
from google.colab import files
files.download('lithium_results.xlsx')

## Flow Graph — Network diagram for a selected year

- **Nodes** = `(stage, tech, region)` — laid out in a 2×3 grid (stage columns, region rows)
- **Solid arrows** = within-region flows (Mine → Processing, same region). Width ∝ flow volume.
- **Dashed arrows** = trade flows between regions. Only arcs with flow > 0 are drawn.
- **Node size** = total production at that node in the selected year.
- **Node color** = region (US=blue, China=red, Australia=green).


In [ ]:
def draw_network(year, min_flow=0.1):
    """
    Draw the supply chain network for a given year.
    Only arcs with flow > min_flow are shown.
    """
    # ── Fixed node positions: x=stage column, y=region row ──────────────
    stage_x  = {'mine': 0,   'proc': 3}
    tech_y   = {'spod': 1.5, 'brine': 0.5, 'loh': 1.5, 'lce': 0.5}
    region_y = {'US': 4,    'China': 0,   'Australia': 2}

    pos = {}
    for (s,t,r) in N:
        pos[(s,t,r)] = (stage_x[s], region_y[r] + tech_y[t])

    # ── Node attributes ──────────────────────────────────────────────────
    region_color = {'US':'#4878CF','China':'#D65F5F','Australia':'#6ACC65'}
    node_colors  = [region_color[r] for (s,t,r) in N]
    node_sizes   = [max(300, prod[s,t,r,year].X * 8) for (s,t,r) in N]
    node_labels  = {(s,t,r): f'{t}\n{r}\n{prod[s,t,r,year].X:.0f}kt' for (s,t,r) in N}

    G = nx.DiGraph()
    G.add_nodes_from(N)

    # ── Collect within-region arcs with flow > threshold ─────────────────
    within_edges   = []
    within_widths  = []
    within_labels  = {}
    for wa_idx,arc in enumerate(within_arcs):
        s_f,t_f,r_f,s_t,t_t,r_t = arc
        v = f_w[wa_idx,year].X
        if v > min_flow:
            frm=(s_f,t_f,r_f); to=(s_t,t_t,r_t)
            within_edges.append((frm,to))
            within_widths.append(max(1, v/5))
            within_labels[(frm,to)] = f'{v:.1f}'

    # ── Collect trade arcs with flow > threshold ──────────────────────────
    trade_edges  = []
    trade_widths = []
    trade_labels = {}
    for ta_idx,arc in enumerate(trade_arcs):
        s_f,t_f,r_f,s_t,t_t,r_t = arc
        v = f_t[ta_idx,year].X
        if v > min_flow:
            frm=(s_f,t_f,r_f); to=(s_t,t_t,r_t)
            trade_edges.append((frm,to))
            trade_widths.append(max(1, v/5))
            trade_labels[(frm,to)] = f'{v:.1f}'

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.set_title(f'Lithium Supply Chain Network — Year {year}\n'
                 f'Node size ∝ production (kt)  |  Arrow width ∝ flow (kt)',
                 fontsize=13, fontweight='bold', pad=15)

    # Draw nodes
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=node_colors,
                           node_size=node_sizes,
                           alpha=0.9)
    nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax,
                            font_size=7, font_color='white', font_weight='bold')

    # Draw within-region arcs (solid, dark grey)
    if within_edges:
        nx.draw_networkx_edges(G, pos, edgelist=within_edges,
                               width=within_widths,
                               edge_color='#333333',
                               arrows=True,
                               arrowsize=20,
                               connectionstyle='arc3,rad=0.0',
                               ax=ax)
        nx.draw_networkx_edge_labels(G, pos, edge_labels=within_labels,
                                     font_size=7, font_color='#333333', ax=ax)

    # Draw trade arcs (dashed, orange)
    if trade_edges:
        nx.draw_networkx_edges(G, pos, edgelist=trade_edges,
                               width=trade_widths,
                               edge_color='#E07B00',
                               style='dashed',
                               arrows=True,
                               arrowsize=18,
                               connectionstyle='arc3,rad=0.25',
                               ax=ax)
        nx.draw_networkx_edge_labels(G, pos, edge_labels=trade_labels,
                                     font_size=7, font_color='#E07B00', ax=ax)

    # Stage column labels
    for s, x in stage_x.items():
        ax.text(x, 6.3, s.upper(), ha='center', va='center',
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', fc='#f0f0f0', ec='grey'))

    # Legend
    legend_elements = [
        mpatches.Patch(color='#4878CF', label='US'),
        mpatches.Patch(color='#D65F5F', label='China'),
        mpatches.Patch(color='#6ACC65', label='Australia'),
        Line2D([0],[0], color='#333333', linewidth=2,            label='Within-region flow'),
        Line2D([0],[0], color='#E07B00', linewidth=2, linestyle='dashed', label='Trade flow'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9,
              framealpha=0.9, title='Legend', title_fontsize=9)

    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'network_{year}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: network_{year}.png')

# Draw for first, middle, and last year
for yr in [2025, 2030, 2035]:
    draw_network(yr)

## Trade Flow Over Time — Bar Chart by Exporter/Importer

In [ ]:
# Aggregate trade flows by year and trade pair
df_trade_pos = df_trade[df_trade['Flow_kt'] > 0.1].copy()
df_trade_pos['Pair'] = df_trade_pos['Exporter'] + ' → ' + df_trade_pos['Importer']
df_pivot = df_trade_pos.groupby(['Year','Pair'])['Flow_kt'].sum().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Mine-stage trade
mine_pairs = [c for c in df_pivot.columns
              if df_trade_pos[df_trade_pos['Pair']==c]['Stage'].iloc[0]=='mine'
              if c in df_pivot.columns]
if mine_pairs:
    df_pivot[mine_pairs].plot(kind='bar', ax=axes[0], width=0.7, colormap='tab10')
    axes[0].set_title('Mine-Stage Trade Flows Over Time', fontweight='bold')
    axes[0].set_xlabel('Year'); axes[0].set_ylabel('Flow (kt)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(title='Trade pair', fontsize=8)

# Right: Proc-stage trade
proc_pairs = [c for c in df_pivot.columns
              if df_trade_pos[df_trade_pos['Pair']==c]['Stage'].iloc[0]=='proc'
              if c in df_pivot.columns]
if proc_pairs:
    df_pivot[proc_pairs].plot(kind='bar', ax=axes[1], width=0.7, colormap='tab20')
    axes[1].set_title('Processing-Stage Trade Flows Over Time', fontweight='bold')
    axes[1].set_xlabel('Year'); axes[1].set_ylabel('Flow (kt)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(title='Trade pair', fontsize=8)

plt.tight_layout()
plt.savefig('trade_flows_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: trade_flows_barchart.png')

## Production Over Time — Stacked Bar by Region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
region_color = {'US':'#4878CF','China':'#D65F5F','Australia':'#6ACC65'}

for ax, stage in zip(axes, ['mine','proc']):
    df_s = df_prod[df_prod['Stage']==stage].groupby(['Year','Region'])['Production_kt'].sum().unstack(fill_value=0)
    df_s[R].plot(kind='bar', stacked=True, ax=ax, width=0.7,
                 color=[region_color[r] for r in R])
    ax.set_title(f'{stage.capitalize()} Production by Region Over Time', fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Production (kt)')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Region', fontsize=9)

plt.tight_layout()
plt.savefig('production_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: production_stacked.png')

## Download All Charts

In [ ]:
from google.colab import files
import os
for fname in ['network_2025.png','network_2030.png','network_2035.png',
              'trade_flows_barchart.png','production_stacked.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')